# 02. Exploratory Data Analysis & Business Insights
**Author:** Renaldy Bilal Setyawan | **Stack:** DuckDB, JupySQL, Python (Pandas)  
**Dataset:** Gayanara E-Commerce (Cleaned)

---

### 📌 Overview
Building on the standardized database from **Phase 1**, this notebook executes **Phase 2 (Exploratory Data Analysis)**. 
Here, we translate raw transactional data into actionable business intelligence, focusing on revenue trends, product performance, and customer purchasing behavior.

> ⚠️ **Note:** This dataset is synthetically generated. Findings marked as data 
> artifacts (Tasks 2 and 7) reflect the generating process, not business behavior.

In [1]:
import duckdb, os

if not os.path.exists('gayanara.db'):
    raise FileNotFoundError(
        "gayanara.db not found. Run 01_Data_Quality_Audit.ipynb first — "
        "it ingests the raw CSVs and creates the database."
    )

con = duckdb.connect('gayanara.db')

# Load the SQL extension and bind the connection
%load_ext sql
%sql con
%config SqlMagic.displaylimit = 50

## 📈 Task 1: Macro Business Metrics (Revenue Trends)
**Business Question:** What is our total net revenue, and how are order volume and average order value trending over time?

In [2]:
%%sql
-- Aggregate to order level first: discount_amount_idr lives on the ORDER,
-- so joining to order_items would repeat it once per line item and
-- overstate total discounts. MAX() collapses it back to one value per order.
WITH order_gross AS (
    SELECT 
        o.order_id,
        DATE_TRUNC('month', o.order_date) AS order_month,
        SUM(oi.quantity) AS units,
        SUM(oi.subtotal_idr) AS gross_revenue,
        MAX(COALESCE(o.discount_amount_idr, 0)) AS discount
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status NOT IN ('cancelled', 'returned')
    GROUP BY o.order_id, order_month, o.discount_amount_idr
)
SELECT 
    order_month,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(units) AS total_units_sold,
    SUM(gross_revenue) AS gross_revenue,
    SUM(discount) AS total_discount,
    SUM(gross_revenue) - SUM(discount) AS net_revenue,
    CAST((SUM(gross_revenue) - SUM(discount)) / COUNT(DISTINCT order_id) AS BIGINT) AS net_aov
FROM order_gross
GROUP BY order_month
ORDER BY order_month ASC;

Running query in 'DuckDBPyConnection'

order_month,total_orders,total_units_sold,gross_revenue,total_discount,net_revenue,net_aov
2022-01-01,38,87,22273000,519690,21753310,572456
2022-02-01,23,53,11457000,211317,11245683,488943
2022-03-01,36,76,21584000,421508,21162492,587847
2022-04-01,35,92,19498000,458972,19039028,543972
2022-05-01,27,52,14658000,384283,14273717,528656
2022-06-01,32,60,10270000,344839,9925161,310161
2022-07-01,34,64,14946000,300164,14645836,430760
2022-08-01,28,59,10721000,441558,10279442,367123
2022-09-01,30,62,17658000,329384,17328616,577621
2022-10-01,33,82,20078000,449238,19628762,594811


> 💡 **Key Business Insights (Revenue Trend):**
> * **Revenue definition:** Net revenue = sum of line-item subtotals minus order-level 
>   discounts, excluding cancelled and returned orders. Shipping is excluded as a 
>   pass-through cost. This reconciles exactly to `orders.total_amount_idr`.
> * **Strong growth:** Monthly net revenue grew from ~17M IDR/month across 2022 to 
>   ~45M IDR/month in 2024, peaking at 68.0M IDR in February 2025 — roughly 3x growth 
>   over three years.
> * **Volume-led, not price-led:** Monthly orders rose from ~32 in 2022 to ~100+ in 
>   2024–25, while AOV shows no directional trend — it fluctuates between roughly 
>   390k and 610k IDR with no upward movement. Early-period swings are amplified by 
>   small monthly order counts (23–39 orders). Growth is driven by customer 
>   acquisition, not by increasing spend per order.
> * **Caveat:** February 2025 is the final month in the dataset and may be partial. 
>   The apparent spike should be confirmed before being read as a trend.

## 🏆 Task 2: Product Performance (Best-Selling Categories)
**Business Question:** Which product categories drive the highest volume, and which drive the highest total revenue?

In [3]:
%%sql
SELECT 
    p.category_clean,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    SUM(oi.quantity) AS total_units_sold,
    SUM(oi.subtotal_idr) AS gross_revenue,
    CAST(SUM(oi.subtotal_idr) / SUM(oi.quantity) AS BIGINT) AS avg_price_per_unit
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.order_status NOT IN ('cancelled', 'returned')
GROUP BY p.category_clean
ORDER BY gross_revenue DESC;

Running query in 'DuckDBPyConnection'

category_clean,total_orders,total_units_sold,gross_revenue,avg_price_per_unit
Jacket,722,1026,223984000,218308
Pants,659,896,221044000,246701
Accessories,694,971,219999000,226570
Dress,614,874,219546000,251197
Shirt,588,824,216616000,262883
T-Shirt,618,851,195579000,229823


> 💡 **Key Business Insights (Category Performance):**
> * **Metric note:** Revenue here is gross (line-item subtotals). Order-level discounts 
>   cannot be attributed to individual categories, so this sums to 1,296.8M IDR gross 
>   rather than the 1,265.3M net headline. The two reconcile exactly.
> * **Jackets lead on both axes:** Jackets top both units sold (1,026) and gross revenue 
>   (224.0M IDR) — despite having the *lowest* average price per unit (218k IDR). Their 
>   revenue lead is purely volume-driven.
> * **Price and revenue are decoupled:** Shirts command the highest average unit price 
>   (263k IDR) but rank 5th in revenue. High price does not compensate for lower volume 
>   in this catalog.
> * **Exceptionally flat distribution:** All six categories fall between 195.6M and 
>   224.0M IDR — a 14% spread between best and worst. No category carries the business, 
>   and none is a clear laggard.
> * **Actionable takeaway:** The flat distribution is unusual and worth interrogating 
>   rather than celebrating. In a real catalog it would suggest either genuinely balanced 
>   demand or an artifact of how the assortment was constructed. Merchandising should 
>   confirm which before treating it as diversification strength.

**Threshold rationale:** Cutoffs are business rules, benchmarked against the observed 
distribution (median recency 145 days; median frequency 4 orders; median lifetime 
spend 1.69M IDR; p90 frequency 6 orders; p90 spend 3.66M IDR):

- **Champions — recency ≤ 60 days:** Deliberately stricter than the median recency of 
  145 days. The intent is a small, high-confidence list marketing can act on, not a 
  balanced split.
- **Champions — frequency ≥ 5 and monetary ≥ 4M IDR:** Frequency 5 sits between the 
  median (4) and p90 (6); the 4M spend threshold sits just above p90 (3.66M). Combined 
  with the recency rule, this yields 26 customers — roughly the top 3%.
- **Lost — recency ≥ 120 days:** Below the median recency of 145 days, so by 
  construction more than half the base falls here (343 of 787). This reflects a real 
  retention problem, but the threshold amplifies it. A 180-day cutoff would produce a 
  more conservative count.

**Limitation:** These are fixed rules, not statistical clusters. The Lost threshold in 
particular is doing definitional work — the segment is large partly because the cutoff 
is set below median recency. A quantile-based RFM scoring approach (1–5 per dimension) 
would adapt as the base grows and avoid this. Fixed thresholds were chosen for 
interpretability by non-technical stakeholders.

## 👥 Task 3: Customer Segmentation (RFM Base Metrics)
**Business Question:** Who are our most valuable customers based on Recency, Frequency, and Monetary (RFM) spending habits?

In [4]:
%%sql
WITH rfm_base AS (
    SELECT 
        c.email,
        CAST((SELECT MAX(order_date) FROM orders) AS DATE) - CAST(MAX(o.order_date) AS DATE) AS recency_days,
        COUNT(DISTINCT o.order_id) AS frequency,
        SUM(o.total_amount_idr) AS monetary_value
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.email
)
SELECT 
    MEDIAN(recency_days) AS median_recency,
    QUANTILE_CONT(frequency, 0.9) AS p90_frequency,
    QUANTILE_CONT(monetary_value, 0.9) AS p90_monetary,
    MEDIAN(frequency) AS median_freq,
    MEDIAN(monetary_value) AS median_monetary
FROM rfm_base;

Running query in 'DuckDBPyConnection'

median_recency,p90_frequency,p90_monetary,median_freq,median_monetary
145.0,6.0,3655976.1999999997,4.0,1690074.0


> 💡 **Key Business Insights (RFM Baselines):**
> * **The base is cold:** Median recency across all 787 customers is 145 days — the 
>   typical customer last purchased nearly five months ago. This is the most important 
>   number in the segmentation: the problem isn't identifying at-risk VIPs, it's that 
>   dormancy is the default state.
> * **High-Value Churn Risk:** Our #1 highest-spending customer has not made a purchase in 129 days. Another top-10 spender hasn't purchased in 229 days.
> * **The Goal:** We need to move away from raw lists and segment these customers into actionable tiers (e.g., "Champions" vs. "At Risk") so the marketing team can run targeted re-engagement campaigns.

In [5]:
%%sql
WITH rfm_base AS (
    SELECT 
        c.email,
        CAST((SELECT MAX(order_date) FROM orders) AS DATE) - CAST(MAX(o.order_date) AS DATE) AS recency_days,
        COUNT(DISTINCT o.order_id) AS frequency,
        SUM(o.total_amount_idr) AS monetary_value
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.email
)
SELECT 
    email,
    recency_days,
    frequency, 
    monetary_value,
    CASE 
        WHEN recency_days <= 60 AND frequency >= 5 AND monetary_value >= 4000000 THEN '🏆 Champions'
        WHEN recency_days > 60 AND (frequency >= 5 OR monetary_value >= 4000000) THEN '⚠️ At Risk (VIPs)'
        WHEN recency_days <= 30 AND frequency < 3 THEN '👋 New / Promising'
        WHEN recency_days >= 120 THEN '💤 Lost / Inactive'
        ELSE '🛒 Regulars'
    END AS customer_segment
FROM rfm_base
ORDER BY monetary_value DESC
LIMIT 15;

Running query in 'DuckDBPyConnection'

email,recency_days,frequency,monetary_value,customer_segment
dewihutapea591@yahoo.com,129,6,7803360,⚠️ At Risk (VIPs)
hadifauzi621@yahoo.com,5,7,6615921,🏆 Champions
yogaiskandar656@gmail.com,56,7,6108037,🏆 Champions
ekosaputra985@gmail.com,154,8,5907000,⚠️ At Risk (VIPs)
putrapermata695@outlook.com,9,10,5733033,🏆 Champions
anggimanurung116@gmail.com,27,8,5109045,🏆 Champions
hanasimbolon349@yahoo.com,229,3,5034443,⚠️ At Risk (VIPs)
dionsiregar730@gmail.com,19,8,4912003,🏆 Champions
vivilestari325@gmail.com,24,9,4865145,🏆 Champions
adityasiregar323@yahoo.com,39,6,4851218,🏆 Champions


> 💡 **Key Business Insights (RFM Segmentation):**
> * **Rule-based segmentation:** Fixed thresholds map customers to named tiers that 
>   marketing can act on directly, at the cost of adaptability (see threshold rationale 
>   above). The trade-off favors interpretability over statistical rigor.
> * **Actionable Takeaway:** Top spenders who have gone dormant (like our #1 customer at 129 days without a purchase) are now correctly flagged as **⚠️ At Risk (VIPs)** instead of Champions. Marketing can instantly export this specific segment to run aggressive, highly-targeted win-back campaigns.

## 🗺️ Task 4: Geospatial Revenue Map & AOV (Hidden Gems)
**Business Question:** Which cities deserve more ad budget? Are there high-value markets outside the major metropolitan areas that yield a higher Average Order Value (AOV)?

In [6]:
%%sql
SELECT 
    shipping_province,
    shipping_city,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(total_amount_idr) AS total_revenue,
    CAST(SUM(total_amount_idr) / COUNT(DISTINCT order_id) AS BIGINT) AS average_order_value
FROM orders
WHERE order_status NOT IN ('cancelled', 'returned')
GROUP BY shipping_province, shipping_city
ORDER BY total_revenue DESC
LIMIT 15;

Running query in 'DuckDBPyConnection'

shipping_province,shipping_city,total_orders,total_revenue,average_order_value
Jawa Tengah,Semarang,141,76571616,543061
Jawa Barat,Depok,150,75974249,506495
Sumatera Utara,Medan,155,75476440,486945
Jawa Barat,Bekasi,151,69672371,461406
Sulawesi Utara,Manado,138,69199639,501447
Sulawesi Selatan,Makassar,138,67318630,487816
Kalimantan Barat,Pontianak,148,67128230,453569
Banten,Tangerang,132,66851296,506449
Jawa Timur,Surabaya,112,65661283,586261
Kalimantan Timur,Balikpapan,124,64958371,523858


> 💡 **Key Business Insights (Geospatial Analysis):**
> * **Top Revenue Driver:** Semarang (Jawa Tengah) leads the pack in total revenue (~76.5M IDR), closely followed by Depok and Medan.
> * **The "Hidden Gems":** **Surabaya (Jawa Timur)** and **Banjarmasin (Kalimantan Selatan)** are the standout markets for profitability. Despite having lower order volumes (112 and 105 respectively), they boast the highest Average Order Values in the top 15 (Surabaya at 586k IDR and Banjarmasin at 583k IDR).
> * **Actionable Takeaway:** Marketing should allocate specific, high-end product ad spend to Surabaya and Banjarmasin, as customers in these regions are clearly willing to add more expensive items (or larger quantities) to their carts.

## 💸 Task 5: Promo Code Impact Evaluation
**Business Question:** Do promo codes actually incentivize higher spending (higher AOV), or are we just giving away discounts to people who would have bought anyway?

In [7]:
%%sql
SELECT 
    CASE 
        WHEN promo_code IS NULL THEN 'No Promo' 
        ELSE 'Used Promo' 
    END AS promo_status,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(total_amount_idr) AS net_revenue,
    SUM(COALESCE(discount_amount_idr, 0)) AS total_discount_given,
    CAST(SUM(total_amount_idr + COALESCE(discount_amount_idr, 0)) 
         / COUNT(DISTINCT order_id) AS BIGINT) AS gross_aov,
    CAST(SUM(total_amount_idr) / COUNT(DISTINCT order_id) AS BIGINT) AS net_aov
FROM orders
WHERE order_status NOT IN ('cancelled', 'returned')
GROUP BY promo_status;

Running query in 'DuckDBPyConnection'

promo_status,total_orders,net_revenue,total_discount_given,gross_aov,net_aov
Used Promo,1025,496651379,31427621,515199,484538
No Promo,1496,768689000,0,513830,513830


> 💡 **Key Business Insights (Promo Impact):**
> * **Methodology note:** A naive comparison of `total_amount_idr` shows promo orders 
>   with a lower AOV (484k vs 514k IDR). This is misleading — `total_amount_idr` is 
>   recorded *net of discount*, so the discount is being subtracted from promo orders 
>   and then read as smaller baskets. The correct comparison adds the discount back to 
>   measure actual cart size.
> * **No measurable lift:** On a like-for-like gross basis, promo orders average 
>   515k IDR versus 514k IDR for organic orders — a 0.3% difference, effectively zero.
> * **The real problem:** The company spent 31.4M IDR on discounts and generated no 
>   incremental basket growth. Promos are subsidizing purchases that would likely have 
>   happened anyway, rather than driving additional spend.
> * **Actionable Takeaway:** Shift from unconditional flat discounts to threshold-based 
>   promos (e.g., "Save 50k on orders above 600k"), which structurally require a larger 
>   basket to unlock the discount.
> * **Limitation:** This is an observational comparison, not a controlled test. Customers 
>   who choose to use promos may differ systematically from those who don't. A holdout 
>   A/B test would be required to establish true incrementality.

## 🚚 Task 6: Courier Return & Cancellation Rates
**Business Question:** Which couriers have the highest return and cancellation rates? Is there a specific logistics partner causing fulfillment issues?

In [8]:
%%sql
SELECT 
    courier,
    COUNT(order_id) AS total_orders,
    SUM(CASE WHEN order_status = 'returned' THEN 1 ELSE 0 END) AS returned_orders,
    SUM(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
    ROUND(SUM(CASE WHEN order_status = 'returned' THEN 1.0 ELSE 0 END) / COUNT(order_id) * 100, 2) AS return_rate_pct,
    ROUND(SUM(CASE WHEN order_status = 'cancelled' THEN 1.0 ELSE 0 END) / COUNT(order_id) * 100, 2) AS cancel_rate_pct
FROM orders
GROUP BY courier
ORDER BY return_rate_pct DESC;

Running query in 'DuckDBPyConnection'

courier,total_orders,returned_orders,cancelled_orders,return_rate_pct,cancel_rate_pct
Pos Indonesia,161,11,12,6.83,7.45
J&T,896,54,111,6.03,12.39
JNE,845,43,84,5.09,9.94
SiCepat,791,35,77,4.42,9.73
Anteraja,307,8,44,2.61,14.33


> 💡 **Key Business Insights (Courier Evaluation):**
> * **Cancellation spread:** Anteraja shows the highest cancellation rate (14.33%, 
>   44 of 307 orders), against a fleet average near 10%. J&T is second (12.39%).
> * **Sample size caveat:** Anteraja (307 orders) and Pos Indonesia (161 orders) carry 
>   far less volume than J&T (896), JNE (845), and SiCepat (791). With 44 cancellation 
>   events, Anteraja's rate carries meaningful uncertainty — the gap versus J&T is 
>   suggestive, not conclusive.
> * **Counter-signal:** Anteraja has the *lowest* return rate (2.61%) of all five 
>   couriers. Whatever drives cancellations does not appear to affect delivery quality, 
>   which points toward pre-dispatch issues (slow pickup, poor tracking) rather than 
>   handling damage.
> * **Actionable takeaway:** Rather than suspending a partner on this evidence, audit 
>   Anteraja's pickup-to-dispatch times and run a controlled volume shift in one region, 
>   measuring cancellation rate before and after. Escalate to contract renegotiation 
>   only if the gap persists at higher volume.

## 🔄 Task 7: Cohort Retention Analysis
**Business Question:** How loyal are our customers? If a user makes their first purchase in a specific month, what percentage of those users return to buy again in the following months?

In [9]:
%%sql
WITH first_purchases AS (
    -- Step 1: First month a customer made a completed purchase.
    -- Status filter matters: without it, a cancelled order can define
    -- the cohort month, and cancelled orders count as "retention".
    SELECT 
        customer_id,
        DATE_TRUNC('month', MIN(order_date)) AS cohort_month
    FROM orders
    WHERE order_status NOT IN ('cancelled', 'returned')
    GROUP BY customer_id
),
cohort_data AS (
    -- Step 2: All subsequent completed purchase months
    SELECT
        f.cohort_month,
        DATE_TRUNC('month', o.order_date) AS activity_month,
        o.customer_id
    FROM orders o
    JOIN first_purchases f ON o.customer_id = f.customer_id
    WHERE o.order_status NOT IN ('cancelled', 'returned')
),
cohort_sizes AS (
    -- Step 3: Original size of each cohort
    SELECT cohort_month, COUNT(DISTINCT customer_id) AS initial_customers
    FROM first_purchases
    GROUP BY cohort_month
),
retention_counts AS (
    -- Step 4: Month index and active customers per cohort
    SELECT
        c.cohort_month,
        DATE_DIFF('month', c.cohort_month, c.activity_month) AS month_index,
        COUNT(DISTINCT c.customer_id) AS active_customers
    FROM cohort_data c
    GROUP BY c.cohort_month, month_index
)
SELECT
    STRFTIME(r.cohort_month, '%Y-%m') AS cohort_month,
    s.initial_customers,
    r.month_index,
    r.active_customers,
    ROUND((r.active_customers * 100.0) / s.initial_customers, 2) AS retention_pct
FROM retention_counts r
JOIN cohort_sizes s ON r.cohort_month = s.cohort_month
ORDER BY r.cohort_month ASC, r.month_index ASC;

Running query in 'DuckDBPyConnection'

cohort_month,initial_customers,month_index,active_customers,retention_pct
2022-01,37,0,37,100.0
2022-01,37,1,1,2.7
2022-01,37,2,2,5.41
2022-01,37,3,1,2.7
2022-01,37,4,1,2.7
2022-01,37,5,2,5.41
2022-01,37,6,2,5.41
2022-01,37,7,3,8.11
2022-01,37,9,1,2.7
2022-01,37,11,5,13.51


In [10]:
%%sql
WITH first_purchases AS (
    SELECT customer_id, DATE_TRUNC('month', MIN(order_date)) AS cohort_month
    FROM orders WHERE order_status NOT IN ('cancelled', 'returned')
    GROUP BY customer_id
),
cohort_data AS (
    SELECT f.cohort_month, DATE_TRUNC('month', o.order_date) AS activity_month, o.customer_id
    FROM orders o JOIN first_purchases f ON o.customer_id = f.customer_id
    WHERE o.order_status NOT IN ('cancelled', 'returned')
),
cohort_sizes AS (
    SELECT cohort_month, COUNT(DISTINCT customer_id) AS initial_customers
    FROM first_purchases GROUP BY cohort_month
),
retention_counts AS (
    SELECT c.cohort_month,
           DATE_DIFF('month', c.cohort_month, c.activity_month) AS month_index,
           COUNT(DISTINCT c.customer_id) AS active_customers
    FROM cohort_data c GROUP BY c.cohort_month, month_index
)
-- Weighted average retention by month index, pooled across all cohorts
SELECT 
    r.month_index,
    SUM(r.active_customers) AS total_active,
    SUM(s.initial_customers) AS total_cohort_base,
    ROUND(SUM(r.active_customers) * 100.0 / SUM(s.initial_customers), 2) AS avg_retention_pct
FROM retention_counts r
JOIN cohort_sizes s ON r.cohort_month = s.cohort_month
WHERE r.month_index <= 12
GROUP BY r.month_index
ORDER BY r.month_index;

Running query in 'DuckDBPyConnection'

month_index,total_active,total_cohort_base,avg_retention_pct
0,768,768,100.0
1,57,623,9.15
2,53,634,8.36
3,61,633,9.64
4,50,614,8.14
5,56,523,10.71
6,67,649,10.32
7,60,690,8.7
8,47,566,8.3
9,62,658,9.42


> 💡 **Key Business Insights (Cohort Retention):**
> * **Methodology note:** Cohorts are defined on completed orders only. Without the 
>   status filter, a cancelled order can define a customer's cohort month and later 
>   cancelled orders count as retention — inflating the curve. Filtering reduces the 
>   cohort base from 787 to 768 customers (19 customers placed only cancelled or 
>   returned orders).
> * **Retention is flat, not decaying:** Pooled across all cohorts, retention holds 
>   between 8.1% and 12.6% from Month 1 through Month 12, with no downward trend 
>   (M1: 9.15%, M6: 10.32%, M12: 10.40%).
> * **This shape is not realistic:** Genuine e-commerce retention curves fall sharply 
>   in Month 1 and then flatten. A flat curve implies a customer's purchase probability 
>   in Month 12 equals their probability in Month 1 — tenure carries no information. 
>   This is the signature of independent random draws, which is consistent with the 
>   dataset being synthetically generated.
> * **Analytical conclusion:** Because the generating process contains no retention 
>   dynamics, this dataset cannot support conclusions about churn drivers, cohort 
>   quality, or lifecycle interventions. The ~10% monthly repeat rate is a property of 
>   the simulation, not a business finding.
> * **What this would mean on real data:** A flat curve at 10% would still indicate a 
>   business with no loyalty mechanics — customers returning at a constant background 
>   rate rather than developing habit. The intervention would be lifecycle 
>   triggers (post-purchase sequences, replenishment prompts) designed to create 
>   tenure-dependent behavior where none currently exists.

---

## ⚠️ Limitations & Scope

**Data provenance.** Synthetic dataset (ngulikdata). Two findings are artifacts of the 
generating process rather than business signal:
- *Category distribution* — all six categories fall within 14% of each other in revenue. 
  Real apparel catalogs are far more concentrated.
- *Retention curve* — flat at ~10% from Month 1 to Month 12 with no decay, indicating 
  independent random purchase draws rather than customer lifecycle behavior.

**What the data cannot support:**
- *True promo ROI* — no cost of goods or margin data, so only revenue impact is 
  measurable, not profitability. The promo comparison is observational; establishing 
  incrementality requires a holdout test.
- *Courier conclusions at partner level* — Anteraja (307 orders) and Pos Indonesia 
  (161) carry too little volume for confident rate comparison. No significance testing 
  was performed.
- *Churn and lifecycle analysis* — see retention curve above.
- *Customer-level financial accuracy* — 3 duplicate email addresses were identified and 
  deliberately not merged (see Notebook 01, Task 2).

**Definitional choices that shape results:**
- Net revenue excludes cancelled and returned orders, and excludes shipping as a 
  pass-through cost. Reconciles exactly to `orders.total_amount_idr` (1,265,340,379 IDR).
- Category-level revenue is gross (1,296,768,000 IDR), because order-level discounts 
  cannot be attributed to individual line items.
- RFM segment boundaries are fixed business rules, not statistical clusters. The "Lost" 
  threshold (120 days) sits below median recency (145 days), which mechanically places 
  the majority of customers in that segment.

**Scope.** Jan 2022 – Feb 2025. February 2025 may be a partial month; the apparent 
revenue spike should not be read as trend.

## 📤 Data Export for Executive Dashboard (Phase 3)
**Objective:** To ensure high performance in the final Business Intelligence (BI) dashboard, we are exporting the aggregated insights from our Exploratory Data Analysis. Connecting a BI tool directly to pre-calculated CSVs prevents the visualization engine from lagging when processing thousands of raw transaction rows.

The following dataframes are exported for visualization:
1. **Geospatial Revenue:** To map out top-performing cities and identify high-AOV "hidden gems."
2. **Promo ROI:** To visualize the negative AOV impact of the current discount strategy.
3. **Courier Evaluation:** To chart the return and cancellation rates for logistics auditing.
4. **RFM Segments:** To display the distribution of our customer base across active, at-risk, and churned tiers.

In [11]:
import os
os.makedirs('dashboard_exports', exist_ok=True)
print("Export directory ready.")

Export directory ready.


In [12]:
%%sql
-- 1. Exporting Geospatial Data
COPY (
    SELECT 
        shipping_province,
        shipping_city,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(total_amount_idr) AS total_revenue,
        CAST(SUM(total_amount_idr) / COUNT(DISTINCT order_id) AS BIGINT) AS average_order_value
    FROM orders
    WHERE order_status NOT IN ('cancelled', 'returned')
    GROUP BY shipping_province, shipping_city
    ORDER BY total_revenue DESC
) TO 'dashboard_exports/dashboard_geospatial.csv' (HEADER, DELIMITER ',');

-- 2. Exporting Promo Impact Data
COPY (
    SELECT 
        CASE 
            WHEN promo_code IS NULL THEN 'No Promo' 
            ELSE 'Used Promo' 
        END AS promo_status,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(total_amount_idr) AS net_revenue,
        SUM(COALESCE(discount_amount_idr, 0)) AS total_discount_given,
        CAST(SUM(total_amount_idr + COALESCE(discount_amount_idr, 0)) 
             / COUNT(DISTINCT order_id) AS BIGINT) AS gross_aov,
        CAST(SUM(total_amount_idr) / COUNT(DISTINCT order_id) AS BIGINT) AS net_aov
    FROM orders
    WHERE order_status NOT IN ('cancelled', 'returned')
    GROUP BY promo_status
) TO 'dashboard_exports/dashboard_promo_roi.csv' (HEADER, DELIMITER ',');

-- 3. Exporting Courier Evaluation Data
COPY (
    SELECT 
        courier,
        COUNT(order_id) AS total_orders,
        SUM(CASE WHEN order_status = 'returned' THEN 1 ELSE 0 END) AS returned_orders,
        SUM(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
        ROUND(SUM(CASE WHEN order_status = 'returned' THEN 1.0 ELSE 0 END) / COUNT(order_id) * 100, 2) AS return_rate_pct,
        ROUND(SUM(CASE WHEN order_status = 'cancelled' THEN 1.0 ELSE 0 END) / COUNT(order_id) * 100, 2) AS cancel_rate_pct
    FROM orders
    GROUP BY courier
    ORDER BY return_rate_pct DESC
) TO 'dashboard_exports/dashboard_courier_eval.csv' (HEADER, DELIMITER ',');

-- 4. Exporting RFM Segmentation Data
COPY (
    WITH rfm_base AS (
        SELECT 
            c.email,
            CAST((SELECT MAX(order_date) FROM orders) AS DATE) - CAST(MAX(o.order_date) AS DATE) AS recency_days,
            COUNT(DISTINCT o.order_id) AS frequency,
            SUM(o.total_amount_idr) AS monetary_value
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY o.customer_id, c.email
    )
    SELECT 
        email,
        recency_days,
        frequency, 
        monetary_value,
        CASE 
            WHEN recency_days <= 60 AND frequency >= 5 AND monetary_value >= 4000000 THEN '🏆 Champions'
            WHEN recency_days > 60 AND (frequency >= 5 OR monetary_value >= 4000000) THEN '⚠️ At Risk (VIPs)'
            WHEN recency_days <= 30 AND frequency < 3 THEN '👋 New / Promising'
            WHEN recency_days >= 120 THEN '💤 Lost / Inactive'
            ELSE '🛒 Regulars'
        END AS customer_segment
    FROM rfm_base
) TO 'dashboard_exports/dashboard_rfm_segments.csv' (HEADER, DELIMITER ',');

-- 5. Exporting Cohort Retention Data
COPY (
    WITH first_purchases AS (
        SELECT customer_id, DATE_TRUNC('month', MIN(order_date)) AS cohort_month
        FROM orders
        WHERE order_status NOT IN ('cancelled', 'returned')
        GROUP BY customer_id
    ),
    cohort_data AS (
        SELECT f.cohort_month, DATE_TRUNC('month', o.order_date) AS activity_month, o.customer_id
        FROM orders o
        JOIN first_purchases f ON o.customer_id = f.customer_id
        WHERE o.order_status NOT IN ('cancelled', 'returned')
    ),
    cohort_sizes AS (
        SELECT cohort_month, COUNT(DISTINCT customer_id) AS initial_customers
        FROM first_purchases GROUP BY cohort_month
    ),
    retention_counts AS (
        SELECT c.cohort_month,
               DATE_DIFF('month', c.cohort_month, c.activity_month) AS month_index,
               COUNT(DISTINCT c.customer_id) AS active_customers
        FROM cohort_data c
        GROUP BY c.cohort_month, month_index
    )
    SELECT STRFTIME(r.cohort_month, '%Y-%m') AS cohort_month,
           s.initial_customers,
           r.month_index,
           r.active_customers,
           ROUND((r.active_customers * 100.0) / s.initial_customers, 2) AS retention_pct
    FROM retention_counts r
    JOIN cohort_sizes s ON r.cohort_month = s.cohort_month
    ORDER BY r.cohort_month ASC, r.month_index ASC
) TO 'dashboard_exports/dashboard_cohort_retention.csv' (HEADER, DELIMITER ',');

Running query in 'DuckDBPyConnection'

Count
650


In [13]:
%%sql
COPY (
    WITH order_gross AS (
        -- Aggregate to order level first to avoid multiplying order-level discount
        SELECT 
            o.order_id,
            STRFTIME(o.order_date, '%Y-%m') AS order_month,
            SUM(oi.quantity) AS units,
            SUM(oi.subtotal_idr) AS gross_revenue,
            MAX(COALESCE(o.discount_amount_idr, 0)) AS discount
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.order_status NOT IN ('cancelled', 'returned')
        GROUP BY o.order_id, order_month, o.discount_amount_idr
    )
    SELECT 
        order_month,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(units) AS total_units_sold,
        SUM(gross_revenue) AS gross_revenue,
        SUM(discount) AS total_discount,
        SUM(gross_revenue) - SUM(discount) AS net_revenue,
        CAST((SUM(gross_revenue) - SUM(discount)) / COUNT(DISTINCT order_id) AS BIGINT) AS net_aov
    FROM order_gross
    GROUP BY order_month
    ORDER BY order_month ASC
) TO 'dashboard_exports/dashboard_monthly_revenue.csv' (HEADER, DELIMITER ',');

Running query in 'DuckDBPyConnection'

Count
38


**✅ Phase 2 (Exploratory Data Analysis) Complete.** 